# EN3160 Assignment 02: Fitting and Alignment
**Name:** Kavin Sulakshitha Gunasekara  
**Index:** 230211E
**GitHub:** https://github.com/gunasekaraks/YOUR_REPOSITORY



In [4]:
from pathlib import Path
import cv2 as cv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.ndimage import gaussian_laplace, maximum_filter
from scipy.optimize import minimize
from IPython.display import display

plt.rcParams.update({
    "figure.figsize": (6.4, 2.6),
    "figure.dpi": 110,
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "legend.fontsize": 6,
    "savefig.bbox": "tight",
    "figure.constrained_layout.use": True,
})

IMAGE_DIR = Path("images")

def read_image(path, flag=cv.IMREAD_COLOR):
    image = cv.imread(str(path), flag)
    if image is None:
        raise FileNotFoundError(f"Cannot read image: {path}")
    return image

def show_image(ax, image, title="", gray=False):
    if image.ndim == 2 or gray:
        ax.imshow(image, cmap="gray", vmin=0, vmax=255)
    else:
        ax.imshow(cv.cvtColor(image, cv.COLOR_BGR2RGB))
    ax.set_title(title)
    ax.axis("off")

## 1. Sunflower detection using LoG



In [5]:
sunflower = read_image(
    IMAGE_DIR / "the_berry_farms_sunflower_field.jpeg",
    cv.IMREAD_REDUCED_COLOR_4
)
gray = cv.cvtColor(sunflower, cv.COLOR_BGR2GRAY).astype(float) / 255.0

# Initial settings: tune after inspecting the supplied image.
sigmas = np.geomspace(2.0, 20.0, 24)
relative_threshold = 0.18
max_detections = 150

responses = np.stack([
    sigma**2 * gaussian_laplace(gray, sigma)
    for sigma in sigmas
])

# Positive responses detect dark blobs.
# For bright blobs instead, use: responses = -responses
peaks = (
    (responses == maximum_filter(responses, size=(3, 3, 3)))
    & (responses > relative_threshold * responses.max())
)
candidates = np.argwhere(peaks)
scores = responses[peaks]
order = np.argsort(scores)[::-1]

blobs = []
height, width = gray.shape

for k in order:
    scale_index, y, x = candidates[k]
    sigma = sigmas[scale_index]
    radius = np.sqrt(2) * sigma

    # Exclude circles cut off by image boundaries.
    if min(x, y, width - 1 - x, height - 1 - y) < radius:
        continue

    # Greedy suppression of duplicate/strongly overlapping detections.
    if any(np.hypot(x - bx, y - by) < 0.7 * (radius + br)
           for bx, by, br, _, _ in blobs):
        continue

    blobs.append((x, y, radius, sigma, responses[scale_index, y, x]))
    if len(blobs) >= max_detections:
        break

blob_table = pd.DataFrame(
    blobs, columns=["x", "y", "radius_px", "sigma", "response"]
).sort_values("radius_px", ascending=False).reset_index(drop=True)

annotated = sunflower.copy()
for x, y, radius, sigma, score in blobs:
    cv.circle(annotated, (int(x), int(y)), round(radius),
              (0, 255, 0), 1, cv.LINE_AA)

for rank, row in blob_table.head(5).iterrows():
    cv.putText(annotated, str(rank + 1),
               (int(row.x), int(row.y)), cv.FONT_HERSHEY_SIMPLEX,
               0.4, (0, 0, 255), 1, cv.LINE_AA)

fig, axes = plt.subplots(1, 2, figsize=(6.6, 2.5))
show_image(axes[0], sunflower, "Reduced input")
show_image(axes[1], annotated, f"LoG detections: {len(blobs)}")
plt.show()

print(f"Sigma range: {sigmas.min():.1f}–{sigmas.max():.1f} pixels "
      f"({len(sigmas)} scales)")
display(blob_table.head(5).round(3))

FileNotFoundError: Cannot read image: images\the_berry_farms_sunflower_field.jpeg